#### Benchmark for every generative model in the repo.

- **Reconstruction quality** (PSNR / SSIM / LPIPS) — for the autoencoders: VAE, VQ-VAE, dVAE, VQ-GAN.
  Computed on a *fixed* validation set so every model is judged on the same faces.
- **Generation quality** (FID) — for whatever can sample from scratch: VAE `N(0, I)`, VQ-VAE + transformer, DCGAN.


In [1]:
import sys, os
sys.path.append(os.path.abspath('vae'))
sys.path.append(os.path.abspath('gan'))

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

from utils import metrics
from utils.utils import return_device, return_photos

In [2]:
device = return_device()

Device is cuda


In [3]:
PATH_TO_PHOTOS = '../archive/lfw-deepfunneled'

all_photos = return_photos(PATH_TO_PHOTOS)

_, val_photos = train_test_split(all_photos, train_size=0.9, shuffle=True, random_state=42)

val_real = torch.stack([torch.as_tensor(im) for im in val_photos]).permute(0, 3, 1, 2).float() / 255
val_loader = torch.utils.data.DataLoader(val_photos, batch_size=128)

print('val images:', tuple(val_real.shape))

100%|██████████| 13233/13233 [00:12<00:00, 1060.28it/s]


Photos uploaded: 13233
val images: (1324, 3, 64, 64)


In [4]:
@torch.no_grad()
def collect_reconstructions(model, loader, device, in_scale=255.0, out_to01=lambda r: r):
    """Run an autoencoder over the loader; return (originals, reconstructions) as [0,1] (N,3,H,W)."""
    model.eval()
    origs, recons = [], []
    for batch in loader:
        x = batch.permute(0, 3, 1, 2).to(device).float() / in_scale
        out = model(x)
        rec = out[0] if isinstance(out, tuple) else out
        origs.append((x if in_scale == 255.0 else (x + 1) / 2).cpu())
        recons.append(out_to01(rec).clamp(0, 1).cpu())
    return torch.cat(origs), torch.cat(recons)

@torch.no_grad()
def sample_to01(generate_fn, n, batch=128):
    """Call generate_fn(k)->(k,3,H,W) in [0,1] repeatedly until n samples are collected."""
    out = []
    while sum(t.shape[0] for t in out) < n:
        k = min(batch, n - sum(t.shape[0] for t in out))
        out.append(generate_fn(k).clamp(0, 1).cpu())
    return torch.cat(out)[:n]

In [5]:
from vae import TinnyVAE
from vq_vae import TinnyVQVAE
from d_vae import TinnyDVAE
from dcgan import Generator as DCGANGenerator
from transformer_prior import GPT

models = {}
generators = {}

# VAE
vae = TinnyVAE().to(device)
vae.load_state_dict(torch.load('checkpoints/vae_reconstruct.pt', map_location=device, weights_only=True))
vae.eval()
models['VAE'] = vae

# VQ-VAE
vqvae = TinnyVQVAE(n_e=1024, e_dim=8, use_ema=False, normalize=True).to(device)
vqvae.load_state_dict(torch.load('checkpoints/vqvae_reconstruct.pt', map_location=device, weights_only=True))
vqvae.eval()
models['VQ-VAE'] = vqvae

# d-VAE
dvae = TinnyDVAE().to(device)
dvae.load_state_dict(torch.load('checkpoints/dvae_reconstruct.pt', map_location=device, weights_only=True))
dvae.eval()
models['D-VAE'] = dvae

# VQ-GAN
vqgan = TinnyVQVAE(n_e=1024, e_dim=8, use_ema=False, normalize=True).to(device)
vqgan.load_state_dict(torch.load('checkpoints/vqgan_reconstruct.pt', map_location=device, weights_only=True))
vqgan.eval()
models['VQ-GAN'] = vqgan


# VQ-VAE Transformer
gpt = GPT(vocab_size=1025, block_size=256, n_layer=6, n_head=8, n_embd=384, dropout=0.1).to(device)
gpt.load_state_dict(torch.load('checkpoints/vqvae_transformer.pt', map_location=device, weights_only=True))
gpt.eval()

@torch.no_grad()
def gen_vqvae(k):
    start = torch.full((k, 1), vqvae.quantize.n_e, dtype=torch.long, device=device)
    tokens = gpt.generate(start, 256, temperature=1.0, top_k=100, forbidden=[vqvae.quantize.n_e])[:, 1:]
    side = int(round(tokens.shape[1] ** 0.5))
    z_q = vqvae.quantize.embed_code(tokens.reshape(-1))
    return vqvae.decode(z_q, (k, side, side, vqvae.quantize.e_dim))  # [0,1]
generators['VQ-VAE'] = gen_vqvae

# DCGAN
dcgan = DCGANGenerator(latent_dim=100).to(device)
dcgan.load_state_dict(torch.load('checkpoints/dcgan_generation.pt', map_location=device, weights_only=True))
dcgan.eval()

@torch.no_grad()
def gen_dcgan(k):
    z = torch.randn(k, dcgan.latent_dim, 1, 1, device=device)
    return (dcgan(z) + 1) / 2  # [-1,1] -> [0,1]
generators['DC-GAN'] = gen_dcgan

## Reconstruction metrics

PSNR / SSIM / LPIPS on the fixed val set (autoencoders only).

In [6]:
recon_rows = {}
for name, model in models.items():
    origs, recons = collect_reconstructions(model, val_loader, device=device)
    recon_rows[name] = metrics.reconstruction_metrics(origs, recons)

recon_table = pd.DataFrame(recon_rows).T
recon_table = recon_table[['PSNR', 'SSIM', 'LPIPS']] if len(recon_table) else recon_table
recon_table  # PSNR up = better, SSIM up = better, LPIPS down = better

Device is cuda
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/opt/conda/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /opt/conda/lib/python3.12/site-packages/lpips/weights/v0.1/alex.pth


/opt/conda/lib/python3.12/site-packages/lpips/lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(model_path, map_location='cpu'), st

Device is cuda
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /opt/conda/lib/python3.12/site-packages/lpips/weights/v0.1/alex.pth
Device is cuda
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /opt/conda/lib/python3.12/site-packages/lpips/weights/v0.1/alex.pth
Device is cuda
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /opt/conda/lib/python3.12/site-packages/lpips/weights/v0.1/alex.pth


,PSNR,SSIM,LPIPS
VAE,25.448189,0.971744,0.282902
VQ-VAE,29.408073,0.988298,0.079548
D-VAE,27.519903,0.982536,0.163891
VQ-GAN,25.192112,0.972077,0.073977


## Generation metrics (FID)

FID against the real val faces (lower is better).

In [7]:
n_fid = min(2000, len(val_real))
real_for_fid = val_real[:n_fid]

fid_rows = {}
for name, gen_fn in generators.items():
    fake = sample_to01(gen_fn, n_fid)
    fid_rows[name] = {'FID': metrics.fid(real_for_fid, fake, device=device)}

fid_table = pd.DataFrame(fid_rows).T
fid_table  # lower = better

,FID
VQ-VAE,43.519209
DC-GAN,21.744372


## Summary


In [8]:
print('Reconstruction metrics:')
display(recon_table)

print('\nGeneration FID:')
display(fid_table)

Reconstruction metrics:


,PSNR,SSIM,LPIPS
VAE,25.448189,0.971744,0.282902
VQ-VAE,29.408073,0.988298,0.079548
D-VAE,27.519903,0.982536,0.163891
VQ-GAN,25.192112,0.972077,0.073977



Generation FID:


,FID
VQ-VAE,43.519209
DC-GAN,21.744372
